# SQL Overview — PT Kirimin
Tujuan: menulis query operasional dasar sambil menjaga grain hasil. Dataset dibuat sintetis dan sengaja memiliki event shipment lebih dari satu untuk satu order.

> Diverifikasi: Python v3.14.7 — sumber: https://docs.python.org/3/ — tanggal cek: 2026-09-18
> Diverifikasi: PostgreSQL v18.6 — sumber: https://www.postgresql.org/docs/release/ — tanggal cek: 2026-09-18

Notebook memakai SQLite bawaan Python sebagai fallback lokal. Uji ulang query pada PostgreSQL sebelum deployment.

In [ ]:
import sqlite3
import pandas as pd

connection = sqlite3.connect(':memory:')
orders = pd.DataFrame([
    ('O-001', 'C-01', 'Jakarta', 800000, 'paid', '2026-09-01'),
    ('O-002', 'C-02', 'JKT', 120000, 'paid', '2026-09-02'),
    ('O-003', 'C-03', 'Bandung', 450000, 'pending', '2026-09-02'),
    ('O-004', 'C-04', 'jakarta', 650000, 'paid', '2026-09-03'),
    ('O-004', 'C-04', 'jakarta', 650000, 'paid', '2026-09-03'),  # duplicate retry
], columns=['order_id','customer_id','origin_city','order_value_idr','payment_status','order_created_at'])
shipments = pd.DataFrame([
    ('S-001', 'O-001', 'in_transit', '2026-09-01 08:00:00'),
    ('S-002', 'O-001', 'delivered', '2026-09-03 10:00:00'),
    ('S-003', 'O-002', 'created', '2026-09-02 10:00:00'),
    ('S-004', 'O-004', 'in_transit', '2026-09-03 12:00:00'),
], columns=['shipment_id','order_id','status','event_time'])
orders.to_sql('orders', connection, index=False)
shipments.to_sql('shipments', connection, index=False)
orders.head()

## 1. Filter dan ordering
Mulai dari query paling kecil. Perhatikan bahwa `WHERE` bekerja pada baris sebelum hasil diringkas.

In [ ]:
pd.read_sql_query('''
SELECT order_id, origin_city, order_value_idr
FROM orders
WHERE payment_status = 'paid'
ORDER BY order_value_idr DESC
LIMIT 10
''', connection)

## 2. Join langsung dan risiko grain
Query berikut memperlihatkan satu order dapat muncul beberapa kali karena memiliki beberapa shipment event. Jangan langsung menjumlahkan `order_value_idr` dari hasil ini.

In [ ]:
joined = pd.read_sql_query('''
SELECT o.order_id, o.order_value_idr, s.status, s.event_time
FROM orders AS o
LEFT JOIN shipments AS s ON s.order_id = o.order_id
ORDER BY o.order_id, s.event_time
''', connection)
joined

## 3. Satu status terbaru per order
Gunakan window function untuk memilih event terbaru. Ini mengembalikan kembali grain satu baris per order.

In [ ]:
latest = pd.read_sql_query('''
WITH ranked AS (
  SELECT s.*,
         ROW_NUMBER() OVER (PARTITION BY order_id ORDER BY event_time DESC) AS rn
  FROM shipments AS s
)
SELECT o.order_id, o.origin_city, o.order_value_idr, r.status AS latest_status
FROM orders AS o
LEFT JOIN ranked AS r ON r.order_id = o.order_id AND r.rn = 1
''', connection)
latest

## 4. Validasi kualitas sederhana
Sebelum memakai metrik, cek duplicate key dan jumlah baris.

In [ ]:
duplicate_keys = pd.read_sql_query('''
SELECT order_id, COUNT(*) AS row_count
FROM orders
GROUP BY order_id
HAVING COUNT(*) > 1
''', connection)
print('duplicate order_id')
display(duplicate_keys)
print('total raw rows:', len(orders))
print('distinct order_id:', orders['order_id'].nunique())

## Mini-exercise
1. Ubah query `latest` agar hanya menampilkan order paid yang status terbarunya bukan `delivered`.
2. Buat agregasi per `origin_city` dan jelaskan mengapa `Jakarta`, `JKT`, dan `jakarta` belum otomatis menjadi satu kota.

Gunakan komentar `# TODO:` saat menulis solusi di notebook Anda.

## Takeaway
SQL yang dapat dipercaya dimulai dari definisi grain, bukan dari menulis query panjang. Lanjutkan dengan assignment untuk membuat `submission.sql`.